# 04. 1차 모델링 : MLR


# 작업 순서

1. 다중회귀분석(MLR) 진행
2. VIF 점검 및 변수 제거
3. 모델 성능 Test
4. MLR 예측값 저장

# 0. 데이터 업로드

활용 데이터셋: dataset_final_ML1조.csv

In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats

!sudo apt-get install -y fonts-nanum
!sudo fc-cache -fv
!rm ~/.cache/matplotlib -rf

plt.rc('font', family='NanumBarunGothic')
plt.rcParams['axes.unicode_minus'] = False

from google.colab import files
print("▶ dataset_final_ML1조 파일 업로드")
uploaded = files.upload()
file_name = list(uploaded.keys())[0]

df = pd.read_csv(file_name)

train_df = df[df['data_split'] == 'train'].copy()
test_df = df[df['data_split'] == 'test'].copy()

print(f"\n✅ 데이터 로드 완료: Train {train_df.shape[0]}건, Test {test_df.shape[0]}건")

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  fonts-nanum
0 upgraded, 1 newly installed, 0 to remove and 24 not upgraded.
Need to get 10.3 MB of archives.
After this operation, 34.1 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 fonts-nanum all 20200506-1 [10.3 MB]
Fetched 10.3 MB in 0s (28.6 MB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Selecting previously unselected package fonts-nanum.
(Reading database ... 118332 files and direc

Saving dataset_final_ML1조.csv to dataset_final_ML1조.csv

✅ 데이터 로드 완료: Train 694건, Test 592건


# 1. 다중회귀분석(MLR) 진행

In [2]:
# 타겟 변수 정의
target = '면적당가격_만원'

# 설명변수 피처셋
features_full = [
    'log_전용면적', '층', '주택연령', 'is_basement',
    'SK미래관거리', '보문역거리',
    '전월세구분_전세', '동그룹_안암동', '주택유형_연립',
    '면적라벨_소형','면적라벨_중형'
]

X_train = train_df[features_full]
y_train = train_df[target]

# 모델 적합
X_train_sm = sm.add_constant(X_train)
model_full = sm.OLS(y_train, X_train_sm)
results_full = model_full.fit()

# 결과
print("="*60)
print(" 📊 2차 다중회귀 요약 결과")
print("="*60)
print(results_full.summary())

 📊 2차 다중회귀 요약 결과
                            OLS Regression Results                            
Dep. Variable:               면적당가격_만원   R-squared:                       0.630
Model:                            OLS   Adj. R-squared:                  0.624
Method:                 Least Squares   F-statistic:                     105.7
Date:                Wed, 19 Aug 2026   Prob (F-statistic):          2.92e-139
Time:                        05:41:11   Log-Likelihood:                -4700.3
No. Observations:                 694   AIC:                             9425.
Df Residuals:                     682   BIC:                             9479.
Df Model:                          11                                         
Covariance Type:            nonrobust                                         
                  coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------
const        1537.1373    176.354

# 1-2. 결과 해석

1. 설명력: 63.0%.
2. is_basement를 제외한 나머지 변수들의 P-value < 0.05 가 나오면서 통계적으로 유의함을 입증함.

# 2. 다중공선성 점검 및 변수 제거(후진제거법)



*   '`is_basement`' 제외 후 OLS 결과 확인 및 VIF 점검





In [3]:
# 타겟 및 피처 정의 (is_basement 제외, 나머지 풀 피처셋 유지)
target = '면적당가격_만원'
features_step3 = [
        'log_전용면적', '층', '주택연령',
    'SK미래관거리', '보문역거리',
    '전월세구분_전세', '동그룹_안암동', '주택유형_연립',
    '면적라벨_소형','면적라벨_중형'
]

X_train = train_df[features_step3]
y_train = train_df[target]

X_test = test_df[features_step3]
y_test = test_df[target]

# 모델 적합
X_train_sm = sm.add_constant(X_train)
model_step3 = sm.OLS(y_train, X_train_sm)
results_step3 = model_step3.fit()

# 결과
print("="*60)
print(" 📊 다중선형회귀 요약 결과 (is_basement 제외)")
print("="*60)
print(results_step3.summary())

# VIF(다중공선성) 확인
vif_data = pd.DataFrame()
vif_data["Feature"] = X_train_sm.columns
vif_data["VIF"] = [variance_inflation_factor(X_train_sm.values, i) for i in range(X_train_sm.shape[1])]

print("\n" + "="*60)
print(" 🔍 다중공선성(VIF) 검증 결과")
print("="*60)

print(vif_data[vif_data['Feature'] != 'const'].sort_values('VIF', ascending=False).round(2))

 📊 다중선형회귀 요약 결과 (is_basement 제외)
                            OLS Regression Results                            
Dep. Variable:               면적당가격_만원   R-squared:                       0.630
Model:                            OLS   Adj. R-squared:                  0.625
Method:                 Least Squares   F-statistic:                     116.3
Date:                Wed, 19 Aug 2026   Prob (F-statistic):          3.25e-140
Time:                        05:41:13   Log-Likelihood:                -4700.5
No. Observations:                 694   AIC:                             9423.
Df Residuals:                     683   BIC:                             9473.
Df Model:                          10                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const       1552.49



*   '`면적라벨_소형`' 제외 후 이전 과정 반복



In [4]:
# 타겟 및 피처 정의 (면적라벨_소형 제외, 나머지 풀 피처셋 유지)
target = '면적당가격_만원'
features_step3 = [
        'log_전용면적', '층', '주택연령',
    'SK미래관거리', '보문역거리',
    '전월세구분_전세', '동그룹_안암동', '주택유형_연립',
        '면적라벨_중형'
]

X_train = train_df[features_step3]
y_train = train_df[target]

X_test = test_df[features_step3]
y_test = test_df[target]

# 모델 적합
X_train_sm = sm.add_constant(X_train)
model_step3 = sm.OLS(y_train, X_train_sm)
results_step3 = model_step3.fit()

# 3. 결과
print("="*60)
print(" 📊 다중선형회귀 요약 결과 (is_basement, 면적라벨_소형 제외)")
print("="*60)
print(results_step3.summary())

# 4. VIF(다중공선성) 확인
vif_data = pd.DataFrame()
vif_data["Feature"] = X_train_sm.columns
vif_data["VIF"] = [variance_inflation_factor(X_train_sm.values, i) for i in range(X_train_sm.shape[1])]

print("\n" + "="*60)
print(" 🔍 다중공선성(VIF) 검증 결과")
print("="*60)

print(vif_data[vif_data['Feature'] != 'const'].sort_values('VIF', ascending=False).round(2))

 📊 다중선형회귀 요약 결과 (is_basement, 면적라벨_소형 제외)
                            OLS Regression Results                            
Dep. Variable:               면적당가격_만원   R-squared:                       0.627
Model:                            OLS   Adj. R-squared:                  0.622
Method:                 Least Squares   F-statistic:                     127.6
Date:                Wed, 19 Aug 2026   Prob (F-statistic):          5.64e-140
Time:                        05:41:13   Log-Likelihood:                -4703.6
No. Observations:                 694   AIC:                             9427.
Df Residuals:                     684   BIC:                             9473.
Df Model:                           9                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const     

# 2-2. 결과 해석

1. 집이 커질수록 단가는 급격히 떨어지는 것 확인(log_전용면적 계수 -307.95).

2. 층수가 1층 높아지면 단가는 약 23.9만 원/m² 상승하고, 집이 1년 낡을수록 약 9.9만 원/m² 하락함.

3. 학교(SK미래관)에서 100m 가까워질수록 단가가 약 19.8만 원/m² 비싸지는 것 확인. 반면 보문역에서는 멀어질수록 비싸지는데(+17.3만 원), 이는 역세권보다 '학세권(안암 상권)'의 파워가 이 지역 청년 주거비에 더 치명적으로 작용함을 보여줌.

4. 월세보다 전세로 계약할 때 단가가 평균 85.6만 원/m² 저렴하게 책정됨. (집주인들의 월세 선호 현상으로 해석)

# 3. 모델 성능 Test

In [5]:
X_test_sm = sm.add_constant(X_test)
y_pred = results_step3.predict(X_test_sm)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("="*60)
print(" 🎯 Test 데이터(25~26년) 예측 성능")
print("="*60)
print(f"▶ Test R-squared (설명력): {r2:.4f}")
print(f"▶ Test RMSE (평균 오차액): {rmse:.1f} 만원/m²")
print(f"▶ Test MAE  (절대 오차액): {mae:.1f} 만원/m²")
print("="*60)

 🎯 Test 데이터(25~26년) 예측 성능
▶ Test R-squared (설명력): 0.6492
▶ Test RMSE (평균 오차액): 204.2 만원/m²
▶ Test MAE  (절대 오차액): 159.3 만원/m²


# 4. MLR 예측값 저장

In [6]:
predictions_df = pd.DataFrame({
    'data_split': test_df['data_split'].values,
    'row_id': test_df.index,
    'actual': y_test.values,
    'MLR_pred': y_pred,
})

predictions_df.to_csv('MLR_predictions.csv', index=False)

from google.colab import files
files.download('MLR_predictions.csv')

predictions_df.head()

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,data_split,row_id,actual,MLR_pred
694,test,694,734.693878,1044.946512
695,test,695,1385.615671,1040.259775
696,test,696,1182.658444,1071.742682
697,test,697,426.621160,393.193248
698,test,698,984.638597,888.755987
